# DeepConvLSTM Replication Notebook (Dual-Track PTQ/QAT)

This notebook replicates the DeepConvLSTM pipeline aligned with the **Efficient human activity recognition on edge devices using DeepConv LSTM architectures** paper using this repo's implementation.

## Scope
- Paper-metric replication track: host-side TFLite evaluation for PTQ/QAT.
- Strict Nano deployability track: full-integer I/O + TFLM compatibility checks.
- QAT section is enabled by default (`RUN_QAT=True`) and can be disabled if needed.
- Both split protocols: `random_stratified` and `user_holdout`.
- Two run modes:
  - `quick`: short sanity execution.
  - `full`: full-fidelity replication settings.

This notebook reuses `src/` modules directly to avoid logic drift.


In [1]:
from pathlib import Path
import sys
import os
import json
import copy
import importlib.util

# Toggle this before running the cell if your kernel crashes on GPU.
USE_GPU = True
if not USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# --- TensorFlow GPU/XLA runtime guardrails (must run before importing project modules) ---
# In this conda env, NVIDIA CUDA libs and ptxas may live under site-packages/nvidia/*.
py_major, py_minor = sys.version_info[:2]
nvidia_root = Path(sys.prefix) / f"lib/python{py_major}.{py_minor}/site-packages/nvidia"
cuda_nvcc_root = nvidia_root / "cuda_nvcc"
ptxas_bin_dir = cuda_nvcc_root / "bin"
ptxas_path = ptxas_bin_dir / "ptxas"

if USE_GPU and nvidia_root.exists():
    # Ensure CUDA shared libraries (including libnvrtc) are discoverable.
    lib_dirs = [p for p in sorted(nvidia_root.glob("*/lib")) if p.is_dir()]
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    ld_entries = existing_ld.split(":") if existing_ld else []
    for lib_dir in reversed(lib_dirs):
        s = str(lib_dir)
        if s not in ld_entries:
            ld_entries.insert(0, s)
    os.environ["LD_LIBRARY_PATH"] = ":".join(ld_entries)

    # Ensure ptxas is on PATH for XLA CUDA compilation.
    if ptxas_path.exists():
        path_entries = os.environ.get("PATH", "").split(":") if os.environ.get("PATH") else []
        if str(ptxas_bin_dir) not in path_entries:
            os.environ["PATH"] = f"{ptxas_bin_dir}:{os.environ.get('PATH', '')}".strip(":")

        # Hint XLA where CUDA toolchain data lives.
        xla_flag = f"--xla_gpu_cuda_data_dir={cuda_nvcc_root}"
        existing_xla = os.environ.get("XLA_FLAGS", "")
        if xla_flag not in existing_xla:
            os.environ["XLA_FLAGS"] = (existing_xla + " " + xla_flag).strip()

# # Avoid greedy VRAM reservation in notebook sessions.
# os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
    "yaml": "PyYAML",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow",
    "tensorflow_model_optimization": "tensorflow-model-optimization",
}
missing = [f"{mod} (pip package: {pkg})" for mod, pkg in required_modules.items() if importlib.util.find_spec(mod) is None]
if missing:
    missing_text = "\n- ".join(missing)
    raise ModuleNotFoundError(
        "Missing required notebook dependencies:\n- "
        + missing_text
        + "\n\nActivate tinymlproj and install dependencies:\n"
        + "conda activate tinymlproj && conda env update -n tinymlproj -f environment.yml --prune"
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import tensorflow as tf

# Configure TF GPU memory growth early (before any real TF GPU allocations happen)
if USE_GPU:
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception as e:
            print(f"[WARN] Could not set memory growth for {g}: {e}")

print("TF version:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# Ensure repo root is importable when launched from different working directories.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_yaml
from src.utils.runtime import check_tensorflow_runtime
from src.utils.repro import load_json, set_global_seed
from src.utils.artifacts import baseline_ckpt_path, history_path, split_npz_path

from src.data.load_wisdm import load_wisdm_dataframe
from src.data.preprocess_zhou2025 import preprocess_zhou2025
from src.data.build_dataset import build_dataset_for_protocol

from src.train.train_baseline import train_baseline_for_protocol
from src.eval.eval_baseline import evaluate_baseline_for_protocol

from src.quant.ptq_full_int8 import quantize_ptq_for_protocol
from src.quant.qat_train import qat_for_protocol
from src.eval.eval_tflite import evaluate_tflite

nvrtc_candidates = []
if nvidia_root.exists():
    for lib_dir in sorted(nvidia_root.glob("*/lib")):
        nvrtc_candidates.extend(sorted(lib_dir.glob("libnvrtc.so*")))

if USE_GPU and not nvrtc_candidates:
    raise RuntimeError(
        "GPU mode requested but libnvrtc.so was not found in tinymlproj. "
        "Install/refresh env with: conda activate tinymlproj && "
        "conda env update -n tinymlproj -f environment.yml --prune"
    )


print(f"Repo root: {REPO_ROOT}")
print(f"USE_GPU: {USE_GPU}")
print(f"nvidia_root exists: {nvidia_root.exists()} at {nvidia_root}")
print(f"ptxas found: {ptxas_path.exists()} at {ptxas_path if ptxas_path.exists() else 'N/A'}")
print(f"libnvrtc candidates: {[str(p) for p in nvrtc_candidates[:3]]}")
print(f"XLA_FLAGS: {os.environ.get('XLA_FLAGS', '')}")

2026-03-02 01:01:02.311815: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-02 01:01:02.311870: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-02 01:01:02.311913: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-02 01:01:02.321609: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 01:01:03.633246: I tensorflow/compiler/

TF version: 2.14.1
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Repo root: /home/dellio/github/har-mcu
USE_GPU: True
nvidia_root exists: True at /home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia
ptxas found: True at /home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia/cuda_nvcc/bin/ptxas
libnvrtc candidates: ['/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.11.2']
XLA_FLAGS: --xla_gpu_cuda_data_dir=/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia/cuda_nvcc


In [2]:
CONFIG_PATH = REPO_ROOT / "configs/default.yaml"
cfg_default = load_yaml(CONFIG_PATH)

if not USE_GPU:
    cfg_default = copy.deepcopy(cfg_default)
    cfg_default.setdefault("env", {})["require_gpu"] = False

runtime_status = check_tensorflow_runtime(cfg_default)
display(pd.DataFrame([runtime_status]))

if "error" in runtime_status:
    raise RuntimeError(runtime_status["error"])

print("Loaded config:", CONFIG_PATH)



,tensorflow_ok,version_ok,gpu_ok,tensorflow_version,gpus
0,True,True,True,2.14.1,[/physical_device:GPU:0]


Loaded config: /home/dellio/github/har-mcu/configs/default.yaml


In [3]:
# Parameters
RUN_MODE = "full"  # "quick" or "full"
WINDOW_SIZE = 100
PROTOCOLS = ["random_stratified", "user_holdout"]
RUN_QAT = True
FORCE_RETRAIN = False
STRICT_FULL_INT8 = True
REQUIRE_TFLM_COMPAT = True
FAIL_ON_PTQ_ERROR = False
FAIL_ON_QAT_ERROR = False
ACCEPTED_INTEGER_IO_DTYPES = ("int8", "uint8")
AUTHOR_STYLE_REP_SAMPLES = 100

print({
    "RUN_MODE": RUN_MODE,
    "WINDOW_SIZE": WINDOW_SIZE,
    "PROTOCOLS": PROTOCOLS,
    "RUN_QAT": RUN_QAT,
    "FORCE_RETRAIN": FORCE_RETRAIN,
    "STRICT_FULL_INT8": STRICT_FULL_INT8,
    "REQUIRE_TFLM_COMPAT": REQUIRE_TFLM_COMPAT,
    "FAIL_ON_PTQ_ERROR": FAIL_ON_PTQ_ERROR,
    "FAIL_ON_QAT_ERROR": FAIL_ON_QAT_ERROR,
    "ACCEPTED_INTEGER_IO_DTYPES": ACCEPTED_INTEGER_IO_DTYPES,
    "AUTHOR_STYLE_REP_SAMPLES": AUTHOR_STYLE_REP_SAMPLES,
})


{'RUN_MODE': 'full', 'WINDOW_SIZE': 100, 'PROTOCOLS': ['random_stratified', 'user_holdout'], 'RUN_QAT': True, 'FORCE_RETRAIN': False, 'STRICT_FULL_INT8': True, 'REQUIRE_TFLM_COMPAT': True, 'FAIL_ON_PTQ_ERROR': False, 'FAIL_ON_QAT_ERROR': False, 'ACCEPTED_INTEGER_IO_DTYPES': ('int8', 'uint8'), 'AUTHOR_STYLE_REP_SAMPLES': 100}


In [4]:
def build_notebook_cfg(base_cfg, run_mode, window_size, protocols, strict_full_int8=True, require_tflm_compatible=True):
    cfg = copy.deepcopy(base_cfg)
    cfg["window_size_default"] = int(window_size)
    cfg["split_protocols"] = list(protocols)

    if run_mode not in {"quick", "full"}:
        raise ValueError("RUN_MODE must be 'quick' or 'full'")

    quant_cfg = cfg.setdefault("quant", {})

    ptq_cfg = quant_cfg.setdefault("ptq", {})
    ptq_cfg["strict_full_int8"] = bool(strict_full_int8)
    ptq_cfg["require_tflm_compatible"] = bool(require_tflm_compatible)
    ptq_cfg["enforce_full_int8"] = True
    ptq_cfg["representative_source"] = "train"
    ptq_cfg["accepted_integer_io_dtypes"] = list(ACCEPTED_INTEGER_IO_DTYPES)
    ptq_cfg.pop("allow_select_tf_ops_fallback", None)

    qat_cfg = quant_cfg.setdefault("qat", {})
    qat_cfg["enabled"] = bool(qat_cfg.get("enabled", True))
    qat_cfg["strict_full_int8"] = bool(strict_full_int8)
    qat_cfg["require_tflm_compatible"] = bool(require_tflm_compatible)
    qat_cfg["enforce_full_int8"] = True
    qat_cfg["representative_source"] = "train"
    qat_cfg["accepted_integer_io_dtypes"] = list(ACCEPTED_INTEGER_IO_DTYPES)

    if run_mode == "quick":
        cfg.setdefault("smoke", {})["enabled"] = True
        cfg["train"]["epochs"] = 1
        cfg["smoke"]["max_windows_per_class"] = 200
        cfg["quant"]["ptq"]["representative_samples"] = 32
        cfg["quant"]["qat"]["representative_samples"] = 32
        cfg["quant"]["qat"]["epochs"] = 1
    else:
        cfg.setdefault("smoke", {})["enabled"] = False
        cfg["smoke"]["max_windows_per_class"] = None
        cfg["quant"]["qat"].setdefault("representative_samples", cfg["quant"]["ptq"].get("representative_samples", 256))

    return cfg

cfg_effective = build_notebook_cfg(
    cfg_default,
    RUN_MODE,
    WINDOW_SIZE,
    PROTOCOLS,
    strict_full_int8=STRICT_FULL_INT8,
    require_tflm_compatible=REQUIRE_TFLM_COMPAT,
)
set_global_seed(int(cfg_effective["seed"]))

CALIBRATION_VARIANTS = [
    {
        "variant": "traincal",
        "representative_source": "train",
        "representative_samples": int(cfg_effective["quant"]["ptq"].get("representative_samples", 256)),
    },
    {
        "variant": "authorcal",
        "representative_source": "test",
        "representative_samples": int(AUTHOR_STYLE_REP_SAMPLES),
    },
]

preview = {
    "window_size_default": cfg_effective["window_size_default"],
    "split_protocols": cfg_effective["split_protocols"],
    "epochs": cfg_effective["train"]["epochs"],
    "ptq_representative_samples": cfg_effective["quant"]["ptq"]["representative_samples"],
    "qat_representative_samples": cfg_effective["quant"]["qat"]["representative_samples"],
    "ptq_strict_full_int8": cfg_effective["quant"]["ptq"]["strict_full_int8"],
    "qat_strict_full_int8": cfg_effective["quant"]["qat"]["strict_full_int8"],
    "ptq_require_tflm_compatible": cfg_effective["quant"]["ptq"]["require_tflm_compatible"],
    "qat_require_tflm_compatible": cfg_effective["quant"]["qat"]["require_tflm_compatible"],
    "accepted_integer_io_dtypes": cfg_effective["quant"]["ptq"]["accepted_integer_io_dtypes"],
    "calibration_variants": CALIBRATION_VARIANTS,
    "smoke_enabled": cfg_effective["smoke"]["enabled"],
    "smoke_max_windows_per_class": cfg_effective["smoke"].get("max_windows_per_class"),
}
print(json.dumps(preview, indent=2))


{
  "window_size_default": 100,
  "split_protocols": [
    "random_stratified",
    "user_holdout"
  ],
  "epochs": 50,
  "ptq_representative_samples": 256,
  "qat_representative_samples": 256,
  "ptq_strict_full_int8": true,
  "qat_strict_full_int8": true,
  "ptq_require_tflm_compatible": true,
  "qat_require_tflm_compatible": true,
  "accepted_integer_io_dtypes": [
    "int8",
    "uint8"
  ],
  "calibration_variants": [
    {
      "variant": "traincal",
      "representative_source": "train",
      "representative_samples": 256
    },
    {
      "variant": "authorcal",
      "representative_source": "test",
      "representative_samples": 100
    }
  ],
  "smoke_enabled": false,
  "smoke_max_windows_per_class": null
}


In [5]:
raw_df, sanity = load_wisdm_dataframe(cfg_effective)
clean_df, pre_stats = preprocess_zhou2025(raw_df, cfg_effective)

class_counts = raw_df["activity"].value_counts().rename("count").reset_index().rename(columns={"index": "activity"})

sanity_table = pd.DataFrame([
    {
        "rows_raw": int(len(raw_df)),
        "rows_after_preprocess": int(len(clean_df)),
        "missing_values": int(sanity["missing_values"]),
        "zero_timestamps": int(sanity["zero_timestamps"]),
        "unique_users": int(raw_df["user"].nunique()),
    }
])

display(sanity_table)
display(class_counts)

paper_readme_rows = 1098207
local_rows = int(len(raw_df))
if local_rows != paper_readme_rows:
    print(
        f"Note: local CSV rows ({local_rows}) differ from README/paper-stated rows ({paper_readme_rows}). "
        "Replication should be interpreted directionally."
    )


,rows_raw,rows_after_preprocess,missing_values,zero_timestamps,unique_users
0,1073623,1073623,0,0,36


,activity,count
0,Walking,417901
1,Jogging,324600
2,Upstairs,122598
3,Downstairs,100192
4,Sitting,59939
5,Standing,48393


Note: local CSV rows (1073623) differ from README/paper-stated rows (1098207). Replication should be interpreted directionally.


In [6]:
dataset_cards = {}
split_rows = []
class_rows = []

for protocol in PROTOCOLS:
    out = build_dataset_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    card = load_json(out["artifacts"]["datacard"])
    dataset_cards[protocol] = card

    split_rows.append(
        {
            "protocol": protocol,
            "window_size": card["windowing"]["window_size"],
            "step": card["windowing"]["step"],
            "candidate_windows": card["windowing"]["candidate_windows"],
            "windows_final": card["windowing"]["windows_final"],
            "train_size": card["split"]["train_size"],
            "val_size": card["split"]["val_size"],
            "test_size": card["split"]["test_size"],
            "split_hash": card["split"]["split_hash"],
        }
    )

    for cls, counts in card["counts"].items():
        class_rows.append(
            {
                "protocol": protocol,
                "activity": cls,
                "train": counts["train"],
                "val": counts["val"],
                "test": counts["test"],
            }
        )

split_df = pd.DataFrame(split_rows)
class_df = pd.DataFrame(class_rows)

display(split_df)
display(class_df)


,protocol,window_size,step,candidate_windows,windows_final,train_size,val_size,test_size,split_hash
0,random_stratified,100,50,21421,20709,11100,2775,6834,e8b3eefc294da8de
1,user_holdout,100,50,21421,20709,10945,2713,7051,8f23a2e13f210a67


,protocol,activity,train,val,test
0,random_stratified,Downstairs,954,238,587
1,random_stratified,Jogging,3425,856,2109
2,random_stratified,Sitting,619,155,381
3,random_stratified,Standing,493,123,304
4,random_stratified,Upstairs,1186,297,730
5,random_stratified,Walking,4423,1106,2723
6,user_holdout,Downstairs,896,250,633
7,user_holdout,Jogging,3159,939,2292
8,user_holdout,Sitting,816,70,269
9,user_holdout,Standing,606,46,268


In [7]:
baseline_train_rows = []

for protocol in PROTOCOLS:
    ckpt_path = baseline_ckpt_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)
    hist_path = history_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)

    if ckpt_path.exists() and not FORCE_RETRAIN:
        train_out = {
            "checkpoint": str(ckpt_path),
            "history": str(hist_path) if hist_path.exists() else None,
            "epochs_ran": 0,
            "final_val_accuracy": None,
            "reused_checkpoint": True,
        }
    else:
        train_out = train_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
        train_out["reused_checkpoint"] = False

    train_out["protocol"] = protocol
    baseline_train_rows.append(train_out)

baseline_train_df = pd.DataFrame(baseline_train_rows)
display(baseline_train_df)


,checkpoint,history,epochs_ran,final_val_accuracy,reused_checkpoint,protocol
0,/home/dellio/github/har-mcu/checkpoints/deepco...,/home/dellio/github/har-mcu/checkpoints/histor...,0,None,True,random_stratified
1,/home/dellio/github/har-mcu/checkpoints/deepco...,/home/dellio/github/har-mcu/checkpoints/histor...,0,None,True,user_holdout


In [8]:
baseline_eval_metrics = {}
baseline_eval_rows = []

for protocol in PROTOCOLS:
    out = evaluate_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    m = load_json(out["metrics_json"])
    baseline_eval_metrics[protocol] = m

    baseline_eval_rows.append(
        {
            "protocol": protocol,
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "confusion_plot": m["confusion_plot"],
            "metrics_json": out["metrics_json"],
            "report_md": out["report_md"],
        }
    )

baseline_eval_df = pd.DataFrame(baseline_eval_rows)
display(baseline_eval_df)


2026-03-02 01:01:06.924615: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 01:01:06.924698: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 01:01:06.924736: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 01:01:07.242956: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 01:01:07.243017: I tensorflow/compile

,protocol,accuracy,macro_f1,confusion_plot,metrics_json,report_md
0,random_stratified,0.511853,0.454694,/home/dellio/github/har-mcu/reports/confusion_...,/home/dellio/github/har-mcu/reports/baseline_T...,/home/dellio/github/har-mcu/reports/baseline_T...
1,user_holdout,0.070770,0.062999,/home/dellio/github/har-mcu/reports/confusion_...,/home/dellio/github/har-mcu/reports/baseline_T...,/home/dellio/github/har-mcu/reports/baseline_T...


In [9]:
ptq_export_metrics = {protocol: {} for protocol in PROTOCOLS}
ptq_eval_metrics = {protocol: {} for protocol in PROTOCOLS}
ptq_rows = []

accepted_set = {str(x).lower() for x in ACCEPTED_INTEGER_IO_DTYPES}

def _strict_ok(export_json):
    in_norm = str(export_json.get("input_dtype_normalized", "")).lower()
    out_norm = str(export_json.get("output_dtype_normalized", "")).lower()
    full_integer_io = bool(export_json.get("full_integer_io", (in_norm in accepted_set and out_norm in accepted_set)))
    return bool(
        export_json.get("status") == "ok"
        and bool(export_json.get("deployable_full_int8", False))
        and bool(export_json.get("tflm_compatible", False))
        and full_integer_io
    )

for protocol in PROTOCOLS:
    for cal in CALIBRATION_VARIANTS:
        variant = cal["variant"]
        rep_source = cal["representative_source"]
        rep_samples = int(cal["representative_samples"])

        cfg_variant = copy.deepcopy(cfg_effective)
        cfg_variant["quant"]["ptq"]["representative_source"] = rep_source
        cfg_variant["quant"]["ptq"]["representative_samples"] = rep_samples

        export_json = {}
        eval_json = None
        eval_report_md = None
        strict_error = None

        try:
            export_out = quantize_ptq_for_protocol(
                cfg_variant,
                WINDOW_SIZE,
                protocol,
                variant=variant,
                raise_on_strict_failure=False,
            )
            export_json = load_json(export_out["report_json"])
        except Exception as exc:
            strict_error = str(exc)

        if not export_json:
            fallback_json_path = (
                Path(cfg_variant["paths"]["reports_dir"])
                / f"ptq_export_T{WINDOW_SIZE}_P{protocol}_{variant}.json"
            )
            if fallback_json_path.exists():
                export_json = load_json(fallback_json_path)

        if export_json:
            ptq_export_metrics[protocol][variant] = export_json
            strict_ok = _strict_ok(export_json)
            strict_error = strict_error or export_json.get("error")

            model_path = export_json.get("tflite_model")
            if model_path and Path(model_path).exists():
                try:
                    eval_out = evaluate_tflite(
                        cfg_variant,
                        model_path,
                        WINDOW_SIZE,
                        protocol,
                        tag=f"ptq_nb_{variant}",
                    )
                    eval_json = load_json(eval_out["metrics_json"])
                    eval_report_md = eval_out["report_md"]
                    ptq_eval_metrics[protocol][variant] = eval_json
                except Exception as eval_exc:
                    if strict_error is None:
                        strict_error = f"Evaluation failed: {eval_exc}"

            ptq_rows.append(
                {
                    "protocol": protocol,
                    "variant": variant,
                    "representative_source": export_json.get("representative_source", rep_source),
                    "representative_samples": export_json.get("representative_samples", rep_samples),
                    "replication_metrics_status": "ok" if eval_json is not None else "error",
                    "strict_deploy_status": "PASS" if strict_ok else "FAIL",
                    "accuracy": eval_json.get("accuracy") if eval_json else None,
                    "macro_f1": eval_json.get("macro_f1") if eval_json else None,
                    "model_size_kb": (
                        eval_json.get("model_size_kb") if eval_json else export_json.get("ptq_tflite_size_kb")
                    ),
                    "input_dtype": export_json.get("input_dtype"),
                    "output_dtype": export_json.get("output_dtype"),
                    "deployable_full_int8": export_json.get("deployable_full_int8", False),
                    "tflm_compatible": export_json.get("tflm_compatible", False),
                    "unsupported_ops": ",".join(export_json.get("unsupported_ops", [])),
                    "has_unidirectional_sequence_lstm": export_json.get("has_unidirectional_sequence_lstm", False),
                    "has_while_op": export_json.get("has_while_op", False),
                    "status": export_json.get("status"),
                    "failure_reason": strict_error,
                    "notes": " | ".join(export_json.get("notes", [])),
                    "tflite_model": export_json.get("tflite_model"),
                    "eval_report_md": eval_report_md,
                }
            )

            if FAIL_ON_PTQ_ERROR and not strict_ok:
                raise RuntimeError(f"PTQ strict gate failed for {protocol}/{variant}: {strict_error}")
        else:
            ptq_rows.append(
                {
                    "protocol": protocol,
                    "variant": variant,
                    "representative_source": rep_source,
                    "representative_samples": rep_samples,
                    "replication_metrics_status": "error",
                    "strict_deploy_status": "FAIL",
                    "accuracy": None,
                    "macro_f1": None,
                    "model_size_kb": None,
                    "input_dtype": None,
                    "output_dtype": None,
                    "deployable_full_int8": False,
                    "tflm_compatible": False,
                    "unsupported_ops": "",
                    "has_unidirectional_sequence_lstm": False,
                    "has_while_op": False,
                    "status": "error",
                    "failure_reason": strict_error or "PTQ export report missing",
                    "notes": "",
                    "tflite_model": None,
                    "eval_report_md": None,
                }
            )
            if FAIL_ON_PTQ_ERROR:
                raise RuntimeError(f"PTQ export missing for {protocol}/{variant}: {strict_error}")

ptq_df = pd.DataFrame(ptq_rows)
display(ptq_df)


INFO:tensorflow:Assets written to: /tmp/tmpmn8rverz/assets


INFO:tensorflow:Assets written to: /tmp/tmpmn8rverz/assets
/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-03-02 01:01:14.760404: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-02 01:01:14.760464: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-02 01:01:14.761161: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpmn8rverz
2026-03-02 01:01:14.767609: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-02 01:01:14.767623: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpmn8rverz
2026-03-02 01:01:14.786138: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 opt

INFO:tensorflow:Assets written to: /tmp/tmpym2ul5xf/assets


INFO:tensorflow:Assets written to: /tmp/tmpym2ul5xf/assets
/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-03-02 01:01:33.284318: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-02 01:01:33.284359: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-02 01:01:33.284490: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpym2ul5xf
2026-03-02 01:01:33.290697: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-02 01:01:33.290712: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpym2ul5xf
2026-03-02 01:01:33.314279: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202

INFO:tensorflow:Assets written to: /tmp/tmpgyt1jml0/assets


INFO:tensorflow:Assets written to: /tmp/tmpgyt1jml0/assets
/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-03-02 01:01:49.656760: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-02 01:01:49.656800: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-02 01:01:49.656919: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpgyt1jml0
2026-03-02 01:01:49.660073: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-02 01:01:49.660086: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpgyt1jml0
2026-03-02 01:01:49.669456: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202

INFO:tensorflow:Assets written to: /tmp/tmppovr_d6k/assets


INFO:tensorflow:Assets written to: /tmp/tmppovr_d6k/assets
/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-03-02 01:02:09.683026: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-02 01:02:09.683068: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-02 01:02:09.683220: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmppovr_d6k
2026-03-02 01:02:09.686538: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-02 01:02:09.686552: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmppovr_d6k
2026-03-02 01:02:09.696130: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202

,protocol,variant,representative_source,representative_samples,replication_metrics_status,strict_deploy_status,accuracy,macro_f1,model_size_kb,input_dtype,...,deployable_full_int8,tflm_compatible,unsupported_ops,has_unidirectional_sequence_lstm,has_while_op,status,failure_reason,notes,tflite_model,eval_report_md
0,random_stratified,traincal,train,256,ok,FAIL,0.510975,0.457947,182.578125,<class 'numpy.int8'>,...,False,False,"EXPAND_DIMS,FILL,WHILE",False,True,failed,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE",Converted with builtin TFLite ops path,/home/dellio/github/har-mcu/models_tflite/deep...,/home/dellio/github/har-mcu/reports/ptq_nb_tra...
1,random_stratified,authorcal,test,100,ok,FAIL,0.510389,0.453583,182.578125,<class 'numpy.int8'>,...,False,False,"EXPAND_DIMS,FILL,WHILE",False,True,failed,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE",Converted with builtin TFLite ops path,/home/dellio/github/har-mcu/models_tflite/deep...,/home/dellio/github/har-mcu/reports/ptq_nb_aut...
2,user_holdout,traincal,train,256,ok,FAIL,0.071905,0.063163,182.578125,<class 'numpy.int8'>,...,False,False,"EXPAND_DIMS,FILL,WHILE",False,True,failed,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE",Converted with builtin TFLite ops path,/home/dellio/github/har-mcu/models_tflite/deep...,/home/dellio/github/har-mcu/reports/ptq_nb_tra...
3,user_holdout,authorcal,test,100,ok,FAIL,0.071196,0.063974,182.578125,<class 'numpy.int8'>,...,False,False,"EXPAND_DIMS,FILL,WHILE",False,True,failed,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE",Converted with builtin TFLite ops path,/home/dellio/github/har-mcu/models_tflite/deep...,/home/dellio/github/har-mcu/reports/ptq_nb_aut...


In [10]:
qat_export_metrics = {protocol: {} for protocol in PROTOCOLS}
qat_eval_metrics = {protocol: {} for protocol in PROTOCOLS}
qat_rows = []

accepted_set = {str(x).lower() for x in ACCEPTED_INTEGER_IO_DTYPES}

def _strict_ok_qat(export_json):
    in_norm = str(export_json.get("input_dtype_normalized", "")).lower()
    out_norm = str(export_json.get("output_dtype_normalized", "")).lower()
    full_integer_io = bool(export_json.get("full_integer_io", (in_norm in accepted_set and out_norm in accepted_set)))
    return bool(
        export_json.get("status") == "ok"
        and bool(export_json.get("deployable_full_int8", False))
        and bool(export_json.get("tflm_compatible", False))
        and full_integer_io
    )

if RUN_QAT:
    for protocol in PROTOCOLS:
        for cal in CALIBRATION_VARIANTS:
            variant = cal["variant"]
            rep_source = cal["representative_source"]
            rep_samples = int(cal["representative_samples"])

            cfg_variant = copy.deepcopy(cfg_effective)
            cfg_variant["quant"]["qat"]["representative_source"] = rep_source
            cfg_variant["quant"]["qat"]["representative_samples"] = rep_samples

            export_json = {}
            eval_json = None
            eval_report_md = None
            strict_error = None

            try:
                qat_out = qat_for_protocol(
                    cfg_variant,
                    WINDOW_SIZE,
                    protocol,
                    variant=variant,
                    raise_on_strict_failure=False,
                )
                export_json = load_json(qat_out["report_json"])
            except Exception as exc:
                strict_error = str(exc)

            if not export_json:
                fallback_json_path = (
                    Path(cfg_variant["paths"]["reports_dir"])
                    / f"qat_export_T{WINDOW_SIZE}_P{protocol}_{variant}.json"
                )
                if fallback_json_path.exists():
                    export_json = load_json(fallback_json_path)

            if export_json:
                qat_export_metrics[protocol][variant] = export_json
                strict_ok = _strict_ok_qat(export_json)
                strict_error = strict_error or export_json.get("error")

                model_path = export_json.get("qat_tflite")
                if model_path and Path(model_path).exists():
                    try:
                        eval_out = evaluate_tflite(
                            cfg_variant,
                            model_path,
                            WINDOW_SIZE,
                            protocol,
                            tag=f"qat_nb_{variant}",
                        )
                        eval_json = load_json(eval_out["metrics_json"])
                        eval_report_md = eval_out["report_md"]
                        qat_eval_metrics[protocol][variant] = eval_json
                    except Exception as eval_exc:
                        if strict_error is None:
                            strict_error = f"Evaluation failed: {eval_exc}"

                qat_rows.append(
                    {
                        "protocol": protocol,
                        "variant": variant,
                        "representative_source": export_json.get("representative_source", rep_source),
                        "representative_samples": export_json.get("representative_samples", rep_samples),
                        "replication_metrics_status": "ok" if eval_json is not None else "error",
                        "strict_deploy_status": "PASS" if strict_ok else "FAIL",
                        "accuracy": eval_json.get("accuracy") if eval_json else None,
                        "macro_f1": eval_json.get("macro_f1") if eval_json else None,
                        "model_size_kb": eval_json.get("model_size_kb") if eval_json else None,
                        "input_dtype": export_json.get("input_dtype"),
                        "output_dtype": export_json.get("output_dtype"),
                        "deployable_full_int8": export_json.get("deployable_full_int8", False),
                        "tflm_compatible": export_json.get("tflm_compatible", False),
                        "unsupported_ops": ",".join(export_json.get("unsupported_ops", [])),
                        "has_unidirectional_sequence_lstm": export_json.get("has_unidirectional_sequence_lstm", False),
                        "has_while_op": export_json.get("has_while_op", False),
                        "status": export_json.get("status"),
                        "failure_reason": strict_error,
                        "notes": " | ".join(export_json.get("notes", [])),
                        "tflite_model": export_json.get("qat_tflite"),
                        "eval_report_md": eval_report_md,
                    }
                )

                if FAIL_ON_QAT_ERROR and not strict_ok:
                    raise RuntimeError(f"QAT strict gate failed for {protocol}/{variant}: {strict_error}")
            else:
                qat_rows.append(
                    {
                        "protocol": protocol,
                        "variant": variant,
                        "representative_source": rep_source,
                        "representative_samples": rep_samples,
                        "replication_metrics_status": "error",
                        "strict_deploy_status": "FAIL",
                        "accuracy": None,
                        "macro_f1": None,
                        "model_size_kb": None,
                        "input_dtype": None,
                        "output_dtype": None,
                        "deployable_full_int8": False,
                        "tflm_compatible": False,
                        "unsupported_ops": "",
                        "has_unidirectional_sequence_lstm": False,
                        "has_while_op": False,
                        "status": "error",
                        "failure_reason": strict_error or "QAT export report missing",
                        "notes": "",
                        "tflite_model": None,
                        "eval_report_md": None,
                    }
                )
                if FAIL_ON_QAT_ERROR:
                    raise RuntimeError(f"QAT export missing for {protocol}/{variant}: {strict_error}")
else:
    print("RUN_QAT=False -> skipping QAT section")

qat_df = pd.DataFrame(qat_rows)
display(qat_df)


,protocol,variant,representative_source,representative_samples,replication_metrics_status,strict_deploy_status,accuracy,macro_f1,model_size_kb,input_dtype,...,deployable_full_int8,tflm_compatible,unsupported_ops,has_unidirectional_sequence_lstm,has_while_op,status,failure_reason,notes,tflite_model,eval_report_md
0,random_stratified,traincal,train,256,error,FAIL,None,None,None,None,...,False,False,,False,False,failed,Unable to construct QAT model: Layer conv1:<cl...,Unable to construct QAT model: Layer conv1:<cl...,None,None
1,random_stratified,authorcal,test,100,error,FAIL,None,None,None,None,...,False,False,,False,False,failed,Unable to construct QAT model: Layer conv1:<cl...,Unable to construct QAT model: Layer conv1:<cl...,None,None
2,user_holdout,traincal,train,256,error,FAIL,None,None,None,None,...,False,False,,False,False,failed,Unable to construct QAT model: Layer conv1:<cl...,Unable to construct QAT model: Layer conv1:<cl...,None,None
3,user_holdout,authorcal,test,100,error,FAIL,None,None,None,None,...,False,False,,False,False,failed,Unable to construct QAT model: Layer conv1:<cl...,Unable to construct QAT model: Layer conv1:<cl...,None,None


In [11]:
TARGET_BASELINE_ACC = 0.9824
TARGET_PTQ_ACC = 0.9709
TARGET_PTQ_SIZE_KB = 136.51

primary_protocol = "random_stratified"
paper_variant = "authorcal"

if primary_protocol not in baseline_eval_metrics:
    raise ValueError("random_stratified baseline results missing; cannot run verdict checks")

baseline_acc_random = float(baseline_eval_metrics[primary_protocol]["accuracy"])
ptq_export_primary = ptq_export_metrics.get(primary_protocol, {}).get(paper_variant)
ptq_eval_primary = ptq_eval_metrics.get(primary_protocol, {}).get(paper_variant)

ptq_replication_available = ptq_eval_primary is not None
strict_ptq_deployable = bool(
    ptq_export_primary
    and ptq_export_primary.get("status") == "ok"
    and ptq_export_primary.get("deployable_full_int8", False)
    and ptq_export_primary.get("tflm_compatible", False)
)

if ptq_replication_available:
    ptq_acc_random = float(ptq_eval_primary["accuracy"])
    ptq_size_random = float(ptq_eval_primary["model_size_kb"])
else:
    ptq_acc_random = None
    ptq_size_random = None

checks = [
    {
        "check": "Baseline accuracy close to paper target",
        "rule": "abs(baseline_acc - 0.9824) <= 0.02",
        "value": baseline_acc_random,
        "status": "PASS" if abs(baseline_acc_random - TARGET_BASELINE_ACC) <= 0.02 else "WARN",
    },
    {
        "check": "PTQ replication metrics available (authorcal)",
        "rule": "PTQ export + host TFLite eval completed",
        "value": "available" if ptq_replication_available else "missing",
        "status": "PASS" if ptq_replication_available else "FAIL",
    },
    {
        "check": "PTQ strict deployment gate (authorcal)",
        "rule": "status=ok, full_integer_io=true, deployable_full_int8=true, tflm_compatible=true",
        "value": (
            f"status={ptq_export_primary.get('status') if ptq_export_primary else None}, "
            f"deployable={ptq_export_primary.get('deployable_full_int8') if ptq_export_primary else None}, "
            f"tflm_compatible={ptq_export_primary.get('tflm_compatible') if ptq_export_primary else None}, "
            f"unsupported_ops={ptq_export_primary.get('unsupported_ops') if ptq_export_primary else None}"
        ),
        "status": "PASS" if strict_ptq_deployable else "FAIL",
    },
]

if ptq_replication_available:
    checks.extend(
        [
            {
                "check": "PTQ accuracy close to paper target",
                "rule": "abs(ptq_acc - 0.9709) <= 0.03",
                "value": ptq_acc_random,
                "status": "PASS" if abs(ptq_acc_random - TARGET_PTQ_ACC) <= 0.03 else "WARN",
            },
            {
                "check": "PTQ size close to paper value",
                "rule": "abs(size_kb - 136.51) <= 60",
                "value": ptq_size_random,
                "status": "PASS" if abs(ptq_size_random - TARGET_PTQ_SIZE_KB) <= 60.0 else "WARN",
            },
        ]
    )

if RUN_QAT:
    qat_export_primary = qat_export_metrics.get(primary_protocol, {}).get(paper_variant)
    qat_eval_primary = qat_eval_metrics.get(primary_protocol, {}).get(paper_variant)
    qat_replication_available = qat_eval_primary is not None
    strict_qat_deployable = bool(
        qat_export_primary
        and qat_export_primary.get("status") == "ok"
        and qat_export_primary.get("deployable_full_int8", False)
        and qat_export_primary.get("tflm_compatible", False)
    )

    checks.extend(
        [
            {
                "check": "QAT replication metrics available (authorcal)",
                "rule": "QAT export + host TFLite eval completed",
                "value": "available" if qat_replication_available else "missing",
                "status": "PASS" if qat_replication_available else "FAIL",
            },
            {
                "check": "QAT strict deployment gate (authorcal)",
                "rule": "status=ok, full_integer_io=true, deployable_full_int8=true, tflm_compatible=true",
                "value": (
                    f"status={qat_export_primary.get('status') if qat_export_primary else None}, "
                    f"deployable={qat_export_primary.get('deployable_full_int8') if qat_export_primary else None}, "
                    f"tflm_compatible={qat_export_primary.get('tflm_compatible') if qat_export_primary else None}, "
                    f"unsupported_ops={qat_export_primary.get('unsupported_ops') if qat_export_primary else None}"
                ),
                "status": "PASS" if strict_qat_deployable else "FAIL",
            },
        ]
    )

verdict_df = pd.DataFrame(checks)
display(verdict_df)


,check,rule,value,status
0,Baseline accuracy close to paper target,abs(baseline_acc - 0.9824) <= 0.02,0.511853,WARN
1,PTQ replication metrics available (authorcal),PTQ export + host TFLite eval completed,available,PASS
2,PTQ strict deployment gate (authorcal),"status=ok, full_integer_io=true, deployable_fu...","status=failed, deployable=False, tflm_compatib...",FAIL
3,PTQ accuracy close to paper target,abs(ptq_acc - 0.9709) <= 0.03,0.510389,WARN
4,PTQ size close to paper value,abs(size_kb - 136.51) <= 60,182.578125,PASS
5,QAT replication metrics available (authorcal),QAT export + host TFLite eval completed,missing,FAIL
6,QAT strict deployment gate (authorcal),"status=ok, full_integer_io=true, deployable_fu...","status=failed, deployable=False, tflm_compatib...",FAIL


In [12]:
summary_rows = []

for protocol in PROTOCOLS:
    if protocol in baseline_eval_metrics:
        m = baseline_eval_metrics[protocol]
        summary_rows.append(
            {
                "protocol": protocol,
                "model": "baseline",
                "variant": "n/a",
                "representative_source": "n/a",
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
                "model_size_kb": None,
                "replication_metrics_status": "ok",
                "strict_deploy_status": "n/a",
                "failure_reason": None,
            }
        )

    for cal in CALIBRATION_VARIANTS:
        variant = cal["variant"]

        export_m = ptq_export_metrics.get(protocol, {}).get(variant)
        eval_m = ptq_eval_metrics.get(protocol, {}).get(variant)
        if export_m:
            strict_ok = bool(
                export_m.get("status") == "ok"
                and export_m.get("deployable_full_int8", False)
                and export_m.get("tflm_compatible", False)
            )
            summary_rows.append(
                {
                    "protocol": protocol,
                    "model": "ptq",
                    "variant": variant,
                    "representative_source": export_m.get("representative_source", cal["representative_source"]),
                    "accuracy": eval_m.get("accuracy") if eval_m else None,
                    "macro_f1": eval_m.get("macro_f1") if eval_m else None,
                    "model_size_kb": (
                        eval_m.get("model_size_kb") if eval_m else export_m.get("ptq_tflite_size_kb")
                    ),
                    "replication_metrics_status": "ok" if eval_m else "error",
                    "strict_deploy_status": "PASS" if strict_ok else "FAIL",
                    "failure_reason": export_m.get("error"),
                }
            )

        if RUN_QAT:
            export_q = qat_export_metrics.get(protocol, {}).get(variant)
            eval_q = qat_eval_metrics.get(protocol, {}).get(variant)
            if export_q:
                strict_ok_q = bool(
                    export_q.get("status") == "ok"
                    and export_q.get("deployable_full_int8", False)
                    and export_q.get("tflm_compatible", False)
                )
                summary_rows.append(
                    {
                        "protocol": protocol,
                        "model": "qat",
                        "variant": variant,
                        "representative_source": export_q.get("representative_source", cal["representative_source"]),
                        "accuracy": eval_q.get("accuracy") if eval_q else None,
                        "macro_f1": eval_q.get("macro_f1") if eval_q else None,
                        "model_size_kb": eval_q.get("model_size_kb") if eval_q else None,
                        "replication_metrics_status": "ok" if eval_q else "error",
                        "strict_deploy_status": "PASS" if strict_ok_q else "FAIL",
                        "failure_reason": export_q.get("error"),
                    }
                )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

reports_dir = Path(cfg_effective["paths"]["reports_dir"])
reports_dir.mkdir(parents=True, exist_ok=True)

summary_csv_path = reports_dir / f"notebook_replication_summary_T{WINDOW_SIZE}.csv"
summary_md_path = reports_dir / f"notebook_replication_summary_T{WINDOW_SIZE}.md"

summary_df.to_csv(summary_csv_path, index=False)

with summary_md_path.open("w", encoding="utf-8") as f:
    f.write(f"# Notebook Replication Summary (T={WINDOW_SIZE})\n\n")
    f.write(f"- Run mode: `{RUN_MODE}`\n")
    f.write(f"- Protocols: {PROTOCOLS}\n")
    f.write(f"- Calibration variants: {[c['variant'] for c in CALIBRATION_VARIANTS]}\n\n")
    f.write("## Summary table\n\n")
    f.write("```\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n```\n\n")
    f.write("## Verdict table\n\n")
    f.write("```\n")
    f.write(verdict_df.to_string(index=False))
    f.write("\n```\n")

print("Saved:")
print("-", summary_csv_path)
print("-", summary_md_path)


,protocol,model,variant,representative_source,accuracy,macro_f1,model_size_kb,replication_metrics_status,strict_deploy_status,failure_reason
0,random_stratified,baseline,n/a,n/a,0.511853,0.454694,NaN,ok,n/a,None
1,random_stratified,ptq,traincal,train,0.510975,0.457947,182.578125,ok,FAIL,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE"
2,random_stratified,qat,traincal,train,NaN,NaN,NaN,error,FAIL,Unable to construct QAT model: Layer conv1:<cl...
3,random_stratified,ptq,authorcal,test,0.510389,0.453583,182.578125,ok,FAIL,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE"
4,random_stratified,qat,authorcal,test,NaN,NaN,NaN,error,FAIL,Unable to construct QAT model: Layer conv1:<cl...
5,user_holdout,baseline,n/a,n/a,0.070770,0.062999,NaN,ok,n/a,None
6,user_holdout,ptq,traincal,train,0.071905,0.063163,182.578125,ok,FAIL,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE"
7,user_holdout,qat,traincal,train,NaN,NaN,NaN,error,FAIL,Unable to construct QAT model: Layer conv1:<cl...
8,user_holdout,ptq,authorcal,test,0.071196,0.063974,182.578125,ok,FAIL,"Unsupported TFLM ops: EXPAND_DIMS, FILL, WHILE"
9,user_holdout,qat,authorcal,test,NaN,NaN,NaN,error,FAIL,Unable to construct QAT model: Layer conv1:<cl...


Saved:
- /home/dellio/github/har-mcu/reports/notebook_replication_summary_T100.csv
- /home/dellio/github/har-mcu/reports/notebook_replication_summary_T100.md


In [13]:
repro_rows = []
DRIFT_TOL = 1e-9

for protocol in PROTOCOLS:
    split_path = split_npz_path(cfg_effective["paths"]["processed_dir"], WINDOW_SIZE, protocol)
    split_hash = None
    if split_path.exists():
        split_npz = np.load(split_path, allow_pickle=True)
        raw_hash = split_npz["split_hash"]
        split_hash = raw_hash.item() if hasattr(raw_hash, "item") else str(raw_hash)

    repeat_out = evaluate_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    repeat_metrics = load_json(repeat_out["metrics_json"])

    old_acc = float(baseline_eval_metrics[protocol]["accuracy"])
    new_acc = float(repeat_metrics["accuracy"])
    drift = abs(new_acc - old_acc)

    repro_rows.append(
        {
            "protocol": protocol,
            "split_hash": split_hash,
            "baseline_acc_first": old_acc,
            "baseline_acc_repeat": new_acc,
            "abs_drift": drift,
            "status": "PASS" if drift <= DRIFT_TOL else "WARN",
        }
    )

repro_df = pd.DataFrame(repro_rows)
display(repro_df)


,protocol,split_hash,baseline_acc_first,baseline_acc_repeat,abs_drift,status
0,random_stratified,e8b3eefc294da8de,0.511853,0.511853,0.0,PASS
1,user_holdout,8f23a2e13f210a67,0.070770,0.070770,0.0,PASS


## Interpretation Notes and Limitations

- This notebook is designed to be **paper-faithful directionally**, while reusing the repository's deterministic script pipeline.
- Dual-track reporting is intentional:
  - `replication_metrics_status` reflects host-side TFLite evaluation availability.
  - `strict_deploy_status` reflects Nano deployment gate readiness.
- A model can have valid quantized I/O and still fail strict deployment if unsupported ops are present for the configured TFLM resolver.
- PTQ/QAT are each run with two calibration profiles:
  - `traincal` (train representative dataset)
  - `authorcal` (authors-style test representative dataset)

## Suggested next steps
- Run `RUN_MODE = "full"` for final replication logs.
- Compare `traincal` vs `authorcal` rows for PTQ/QAT drift in accuracy and model size.
- Use strict PASS rows only for Nano deployment artifacts.
